In [26]:
import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from IPython.display import display

In [27]:
# =========================================================
# 0) READ DATA
# - rows: instance/locator
# - columns: type/feature
# - cell: abundance
# =========================================================

csv_path = "benthic_hellinger_num.csv"  # <-- path to your file
df = pd.read_csv(csv_path, sep=";")

##csv_path = "dune_hellinger.csv"
##df = pd.read_csv(csv_path, sep=",")

##csv_path = "mite_hellinger.csv"
##df = pd.read_csv(csv_path, sep=",")


In [28]:
#numeric_df = df.select_dtypes(include=[np.number])

In [29]:
# Get only the numeric columns.
X = df.select_dtypes(include=[np.number]).copy()
X = X.fillna(0.0)

Xv = X.to_numpy(dtype=float)
n = Xv.shape[0]
if n < 3:
    raise ValueError("En az 3 örnek (satır) gerekli.")


In [30]:
# =========================================================
# 1) DISTANCE FUNCTIONS (pairwise)
# =========================================================
def bray_curtis_distance(x, y):
    num = np.sum(np.abs(x - y))
    den = np.sum(np.abs(x) + np.abs(y))
    return 0.0 if den == 0 else float(num / den)

def kulczynski_distance(x, y):
    sx = np.sum(x)
    sy = np.sum(y)
    m = np.sum(np.minimum(x, y))
    if sx == 0 and sy == 0:
        return 0.0
    if sx == 0 or sy == 0:
        return 1.0
    sim = 0.5 * (m / sx + m / sy)
    return float(1.0 - sim)

def chord_distance(x, y):
    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx == 0 and ny == 0:
        return 0.0
    if nx == 0 or ny == 0:
        return 1.0
    x1 = x / nx
    y1 = y / ny
    return float(np.linalg.norm(x1 - y1))

# Chi-square distance
def chi_square_distance(x, y):
    # Oransal bolluk (pik)
    x_prop = x / np.sum(x) if np.sum(x) != 0 else np.zeros_like(x)
    y_prop = y / np.sum(y) if np.sum(y) != 0 else np.zeros_like(y)
    mean_prop = (x_prop + y_prop) / 2

    with np.errstate(divide='ignore', invalid='ignore'):
        dist = np.where(mean_prop == 0, 0, ((x_prop - y_prop) ** 2) / mean_prop)

    return np.sqrt(np.sum(dist))

# --- NB placeholder ---
def NB_distance(x, y):
    numerator = np.sum(np.abs(x - y) * (np.abs(x) + np.abs(y)))
    denominator = np.sum(x**2 + y**2)
    return 0.0 if denominator == 0 else float(numerator / denominator)

In [31]:
# =========================================================
# 2) GENERATE DISTANCE MATRIX (NxN)
# =========================================================
def pairwise_distance_matrix(Xv, dist_func):
    n = Xv.shape[0]
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            # Tüm fonksiyonlar artık standart olarak (x, y) alıyor
            d = dist_func(Xv[i], Xv[j])
            D[i, j] = d
            D[j, i] = d
    return D

def ensure_distance_matrix(D):
    D = np.asarray(D, dtype=float)
    if D.ndim != 2 or D.shape[0] != D.shape[1]:
        raise ValueError("Distance matrix NxN olmalı.")
    if not np.allclose(D, D.T, atol=1e-10):
        raise ValueError("Distance matrix simetrik olmalı.")
    np.fill_diagonal(D, 0.0)
    if np.any(D < -1e-12):
        raise ValueError("Distance matrix negatif değer içeriyor.")
    return np.maximum(D, 0.0)

# Distance generators
distance_builders = {
    "NB": lambda: pairwise_distance_matrix(Xv, NB_distance),
    "Bray-Curtis": lambda: pairwise_distance_matrix(Xv, bray_curtis_distance),
    "Kulczynski": lambda: pairwise_distance_matrix(Xv, kulczynski_distance),
    "Chord": lambda: pairwise_distance_matrix(Xv, chord_distance),
    "Chi-square": lambda: pairwise_distance_matrix(Xv, chi_square_distance),
}

# Calculate and validate the matrices.
distance_matrices = {name: ensure_distance_matrix(build()) for name, build in distance_builders.items()}

In [32]:
# =========================================================
# 3) Dunn, complete linkage, metrics for all k's
# =========================================================
def dunn_index_from_distance(D, labels):
    D = ensure_distance_matrix(D)
    labels = np.asarray(labels)
    uniq = np.unique(labels)
    if uniq.size < 2:
        return np.nan

    clusters = [np.where(labels == c)[0] for c in uniq]

    # max intracluster diameter
    diameters = []
    for idx in clusters:
        if idx.size <= 1:
            diameters.append(0.0)
        else:
            sub = D[np.ix_(idx, idx)]
            diameters.append(np.max(sub))
    max_diam = np.max(diameters)

    # min intercluster distance
    min_inter = np.inf
    for a in range(len(clusters)):
        for b in range(a + 1, len(clusters)):
            ia, ib = clusters[a], clusters[b]
            sub = D[np.ix_(ia, ib)]
            m = np.min(sub)
            if m < min_inter:
                min_inter = m

    if max_diam <= 1e-15:
        return np.nan
    return float(min_inter / max_diam)

def build_linkage_complete(D):
    condensed = squareform(D, checks=False)
    return linkage(condensed, method="complete")

def labels_for_k(Z, k):
    return fcluster(Z, t=k, criterion="maxclust")

def valid_k(labels, n):
    u = np.unique(labels).size
    return (u >= 2) and (u <= n - 1)

def evaluate_complete_linkage_consensus(X_df, distance_matrices, k_min=2, k_max=12):
    Xv_local = X_df.select_dtypes(include=[np.number]).copy().fillna(0.0).to_numpy(dtype=float)
    n_local = Xv_local.shape[0]
    k_max = min(k_max, n_local - 1)

    rows = []
    label_store = {}

    for dist_name, D in distance_matrices.items():
        Z = build_linkage_complete(D)

        for k in range(k_min, k_max + 1):
            labels = labels_for_k(Z, k)
            label_store[(dist_name, k)] = labels

            if valid_k(labels, n_local):
                sil = silhouette_score(D, labels, metric="precomputed")
                ch = calinski_harabasz_score(Xv_local, labels)
                db = davies_bouldin_score(Xv_local, labels)
            else:
                sil = np.nan
                ch = np.nan
                db = np.nan

            dunn = dunn_index_from_distance(D, labels)

            rows.append({
                "Distance": dist_name,
                "k": k,
                "Silhouette": sil,          # higher better
                "Calinski-H": ch,           # higher better
                "Davies-Bouldi": db,        # lower better
                "Dunn": dunn                # higher better
            })

    long_df = pd.DataFrame(rows)

    # metric ranks within distance
    def rank_within_distance(df_one):
        out = df_one.copy()
        out["r_Silhouette"] = out["Silhouette"].rank(ascending=False, method="average")
        out["r_Dunn"] = out["Dunn"].rank(ascending=False, method="average")
        out["r_CH"] = out["Calinski-H"].rank(ascending=False, method="average")
        out["r_DB"] = out["Davies-Bouldi"].rank(ascending=True, method="average")
        out["rank_mean_metrics"] = out[["r_Silhouette", "r_Dunn", "r_CH", "r_DB"]].mean(axis=1)
        return out

    ranked_df = (
        long_df
        .groupby("Distance", group_keys=False)
        .apply(rank_within_distance)
        .reset_index(drop=True)
    )

    # consensus on k basis
    consensus_df = (
        ranked_df
        .groupby("k", as_index=False)
        .agg(
            consensus_rank=("rank_mean_metrics", "mean"),
            consensus_rank_std=("rank_mean_metrics", "std"),
            mean_Silhouette=("Silhouette", "mean"),
            mean_Dunn=("Dunn", "mean"),
            mean_CH=("Calinski-H", "mean"),
            mean_DB=("Davies-Bouldi", "mean"),
        )
        .sort_values(["consensus_rank", "consensus_rank_std"], ascending=[True, True])
        .reset_index(drop=True)
    )

    best_k = int(consensus_df.loc[0, "k"])
    return long_df, ranked_df, consensus_df, best_k, label_store

In [33]:
# =========================================================
#4) RUN + OUTPUT TABLE
# =========================================================
KMAX = 20
long_df, ranked_df, consensus_df, best_k, labels_dict = evaluate_complete_linkage_consensus(
    X_df=X,
    distance_matrices=distance_matrices,
    k_min=2,
    k_max=KMAX
)

print("CONSENSUS best k =", best_k)

# Screenshot formatında results_df:
results = []
for dist_name in distance_matrices.keys():
    row = long_df[(long_df["Distance"] == dist_name) & (long_df["k"] == best_k)].iloc[0]
    results.append({
        "Distance": dist_name,
        "k_used": best_k,
        "Silhouette": row["Silhouette"],
        "Davies-Bouldi": row["Davies-Bouldi"],
        "Calinski-H": row["Calinski-H"],
        "Dunn": row["Dunn"],
    })

results_df = pd.DataFrame(results)

#In the order shown in the image:
results_df = results_df[["Distance", "k_used", "Silhouette", "Davies-Bouldi", "Calinski-H", "Dunn"]]

# Round:
results_df["Silhouette"] = results_df["Silhouette"].round(3)
results_df["Davies-Bouldi"] = results_df["Davies-Bouldi"].round(3)
results_df["Dunn"] = results_df["Dunn"].round(3)

# CH is sometimes very large; scientific notation:
results_df["Calinski-H"] = results_df["Calinski-H"].apply(lambda v: f"{v:.2E}" if pd.notnull(v) else v)

display(results_df)

# (Optional) Also print the cluster labels for the same k:
labels_bestk_df = pd.DataFrame(
    {d: labels_dict[(d, best_k)] for d in distance_matrices.keys()},
    index=X.index
)
# display(labels_bestk_df.head(20))


CONSENSUS best k = 20


C:\Users\nursu\AppData\Local\Temp\ipykernel_2272\1543900779.py:98: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rank_within_distance)


,Distance,k_used,Silhouette,Davies-Bouldi,Calinski-H,Dunn
0,NB,20,0.205,1.237,1.29E+01,0.275
1,Bray-Curtis,20,0.254,1.384,7.96E+00,0.377
2,Kulczynski,20,0.261,1.479,6.30E+00,0.362
3,Chord,20,0.313,1.406,6.13E+00,0.214
4,Chi-square,20,0.238,1.412,9.20E+00,0.489
